# VWAP Events 04 — fading extreme displacement from VWAP

Protocol: `discussions/VWAP_Events/006_claude_ve04_preregistration.md` (fixed before this run).

Displacement ≥ 3/4/5σ·√15 or ≥ 0.5/0.75/1% from VWAP (any time, 30-min cooldown) → fade toward VWAP → target / VWAP touch / stop / 60–240 min.
**Cell chosen on DEV (before 2026-04-01), judged once on TEST.** G1 hour-matched real − placebo > 0; G2 TEST gross lower bound > 7 bp.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import vwap_events as ve, vwap_fade as vf, vwap_extreme as vx
importlib.reload(vx)
SYNTHETIC=os.getenv('VE04_SYNTHETIC','0')=='1';ASSETS=['BTC','ETH'];xp=vx.XParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='ve04-');OUT=Path(tmp.name)/'04';xp=vx.XParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'VWAP_Events'/'04'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_vwap_extreme.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-300:]);assert chk.returncode==0,'tests failed'
print(json.dumps(vx.registration(xp),indent=1,default=str))

...                                                                      [100%]
3 passed in 6.68s

{
 "k_sigma": [
  3.0,
  4.0,
  5.0
 ],
 "pct": [
  0.005,
  0.0075,
  0.01
 ],
 "stretch_m": 15,
 "warmup_min": 30,
 "cooldown": 30,
 "targets": [
  20,
  40,
  80,
  "vwap"
 ],
 "stops": [
  20,
  40,
  80
 ],
 "holds": [
  60,
  240
 ],
 "min_dev_n": 50,
 "tradeable_bp": 7.0,
 "costs_bp": [
  4.0,
  7.0,
  12.0
 ],
 "n_boot": 2000,
 "status": "PREREGISTERED_VE006",
 "version": "VE04-0.1.0",
 "test_start": "2026-04-01 00:00:00+00:00",
 "selection": "DEV real cell with max day-bootstrap q2.5 of mean gross bp (n_dev >= 50)",
 "gates": {
  "G1": "TEST hour-matched real - placebo > 0, q2.5 > 0",
  "G2": "TEST real gross q2.5 > 7 bp"
 }
}


In [2]:
T0=time.time();parts=[]
for i,a in enumerate(ASSETS):
    df=vf.synthetic_path_klines(n_days=200,seed=4+i,start='2025-10-01') if SYNTHETIC else pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','high','low','close','volume','quote_volume'])
    parts.append(vx.run_asset(df,a,xp));print(a,f'{time.time()-T0:.0f}s')
R=pd.concat(parts,ignore_index=True);print('simulated trades:',len(R))
cnt=R[(R.target=='vwap')&(R.stop==R.stop.min())&(R.hold==R.hold.min())].groupby(['kind','thr','test'],observed=True).size().unstack('test')
print('events per threshold (real / placebo; DEV=False, TEST=True):');display(cnt)

BTC 108s
ETH 227s
simulated trades: 17789688
events per threshold (real / placebo; DEV=False, TEST=True):


test            False  True 
kind    thr                 
placebo k3      78422  25956
        k4      46648  14835
        k5      28241   8692
        p0.5   134514  34243
        p0.75   88427  19667
        p1      60822  12037
real    k3      26066   8917
        k4      15770   5292
        k5       9839   3276
        p0.5    45157  11809
        p0.75   30194   7012
        p1      21009   4392

## Selection on DEV, decision on TEST

In [3]:
dev=vx.dev_table(R,xp);dev.to_csv(OUT/'04_dev_selection.csv',index=False)
display(dev.sort_values('dev_lo',ascending=False).head(10).round(2))
cell=vx.select(dev,xp);print('selected cell:',cell)
res=vx.evaluate(R,cell,xp);print(json.dumps(res,indent=1,default=float))

,thr,target,stop,hold,dev_gross,dev_lo,dev_hi,dev_n,dev_win
66,k5,vwap,20,60,1.23,0.32,2.12,9839,0.34
60,k5,80,20,60,1.09,0.28,1.88,9839,0.34
132,p1,80,20,60,0.86,0.25,1.44,21009,0.28
54,k5,40,20,60,0.68,0.05,1.28,9839,0.39
133,p1,80,20,240,0.78,0.04,1.48,21009,0.22
108,p0.75,80,20,60,0.54,-0.00,1.05,30194,0.28
138,p1,vwap,20,60,0.65,-0.08,1.40,21009,0.27
36,k4,80,20,60,0.56,-0.08,1.20,15770,0.33
84,p0.5,80,20,60,0.31,-0.09,0.73,45157,0.29
126,p1,40,20,60,0.34,-0.10,0.80,21009,0.35


selected cell: {'thr': 'k5', 'target': 'vwap', 'stop': 20, 'hold': 60, 'dev_gross': 1.233174471677581, 'dev_lo': 0.3224333590725082, 'dev_n': 9839}
{
 "version": "VE04-0.1.0",
 "status": "PREREGISTERED_VE006",
 "cell": {
  "thr": "k5",
  "target": "vwap",
  "stop": 20,
  "hold": 60,
  "dev_gross": 1.233174471677581,
  "dev_lo": 0.3224333590725082,
  "dev_n": 9839
 },
 "test_n": 3276,
 "test_gross": 0.7532786222925849,
 "test_lo": -0.3936853844374369,
 "test_hi": 2.02279343495363,
 "placebo_gross": 0.3709686548964262,
 "real_minus_placebo_hour_matched": 0.1898628026901022,
 "diff_lo": -1.088445642583132,
 "diff_hi": 1.4552617819215545,
 "test_win_rate": 0.4062881562881563,
 "G1_vwap_specific": "FAIL",
 "G2_tradeable": "FAIL",
 "test_net_4bp": -3.246721377707415,
 "test_net_7bp": -6.246721377707415,
 "test_net_12bp": -11.246721377707415,
 "verdict": "NO_EDGE"
}


## All cells (descriptive)

In [4]:
FT=vx.full_table(R,xp);FT.to_csv(OUT/'04_all_cells.csv',index=False)
for test in (False,True):
    print('TEST' if test else 'DEV','— real gross bp (rows thr x hold, cols target x stop)')
    display(FT[(FT.kind=='real')&(FT.test==test)].pivot_table(index=['thr','hold'],columns=['target','stop'],values='gross',observed=True).round(1))
print('TEST real - placebo, VWAP-touch exit:')
v=FT[(FT.target=='vwap')&(FT.test)].pivot_table(index=['thr','hold','stop'],columns='kind',values=['gross','win','n'],observed=True)
display(v.round(2))

DEV — real gross bp (rows thr x hold, cols target x stop)


target       20             40             80           vwap          
stop         20   40   80   20   40   80   20   40   80   20   40   80
thr   hold                                                            
k3    60   -0.1 -0.1 -0.2 -0.1 -0.3 -0.6  0.1 -0.2 -0.7 -0.1 -0.3 -0.9
      240  -0.0 -0.1 -0.1 -0.1 -0.5 -0.6 -0.1 -0.7 -0.8 -0.4 -1.0 -1.2
k4    60    0.1  0.2  0.0  0.4  0.2 -0.1  0.6  0.2 -0.1  0.5  0.3 -0.3
      240   0.2  0.1  0.0  0.4  0.2  0.0  0.4 -0.1 -0.4 -0.1 -0.1 -0.7
k5    60    0.3  0.2 -0.2  0.7  0.5 -0.1  1.1  0.9  0.1  1.2  1.0  0.3
      240   0.3  0.0 -0.3  0.5  0.3 -0.3  0.8  0.8  0.4  0.6  0.6  0.5
p0.5  60    0.1  0.0 -0.1  0.2  0.0 -0.4  0.3  0.0 -0.5  0.2 -0.1 -0.6
      240   0.1 -0.1 -0.1  0.2 -0.1 -0.4  0.3 -0.1 -0.7  0.0 -0.5 -1.0
p0.75 60    0.0  0.1 -0.2  0.3  0.1 -0.2  0.5  0.3 -0.3  0.3  0.0 -0.6
      240   0.0  0.1 -0.3  0.2 -0.1 -0.3  0.4  0.1 -0.6  0.2 -0.2 -1.0
p1    60    0.1 -0.1 -0.2  0.3 -0.2 -0.4  0.9  0.0 -0.4  0.6 -0.3 -0.8
      240   0.1 -0.1 -0.4  0.3 -0.2 -0.9  0.8 -0.0 -0.6  0.6 -0.5 -1.2

TEST — real gross bp (rows thr x hold, cols target x stop)


target       20             40             80           vwap          
stop         20   40   80   20   40   80   20   40   80   20   40   80
thr   hold                                                            
k3    60   -0.1 -0.3 -0.1 -0.1 -0.3  0.0  0.0  0.1  0.5  0.2  0.4  1.1
      240  -0.1 -0.3  0.3 -0.5 -0.8  0.1 -0.8 -0.8  0.1 -0.1  0.2  1.7
k4    60   -0.0 -0.2 -0.1 -0.3 -0.6 -0.5  0.1 -0.1  0.0  0.7  0.7  1.0
      240  -0.1 -0.3 -0.3 -0.7 -1.0 -1.2 -0.7 -0.9 -1.3  0.1  0.8  1.3
k5    60    0.3 -0.0 -0.3  0.3  0.1 -0.3  0.2  0.1 -0.7  0.8  0.8  0.3
      240   0.1 -0.1 -0.9 -0.1 -0.1 -1.2 -1.0 -0.7 -2.3 -0.1  0.9  0.9
p0.5  60   -0.0 -0.2 -0.0  0.1 -0.4 -0.1  0.2 -0.2 -0.0  0.1 -0.2  0.2
      240  -0.1 -0.2  0.2 -0.2 -0.6  0.2 -0.1 -0.5  0.4 -0.1 -0.0  1.1
p0.75 60    0.3 -0.1 -0.2  0.5  0.1  0.2  0.5  0.2  0.4  0.8  0.4  1.0
      240   0.3 -0.2 -0.2  0.2 -0.2  0.6 -0.3 -0.5  0.7  0.3  0.5  2.3
p1    60    0.4  0.1  0.6  0.2 -0.1  0.5  0.1 -0.0  0.6  0.3  0.5  1.4
      240   0.4  0.0  0.3 -0.2 -0.6 -0.3 -1.0 -1.1 -0.2 -0.2  0.4  2.5

TEST real - placebo, VWAP-touch exit:


gross              n              win      
kind            placebo  real  placebo     real placebo  real
thr   hold stop                                              
k3    60   20      0.07  0.24  25955.0   8917.0    0.39  0.39
           40      0.03  0.41  25955.0   8917.0    0.50  0.50
           80      0.38  1.10  25955.0   8917.0    0.54  0.54
      240  20     -0.24 -0.14  25948.0   8917.0    0.30  0.31
           40     -0.35  0.17  25948.0   8917.0    0.44  0.44
           80     -0.18  1.73  25948.0   8917.0    0.54  0.54
k4    60   20      0.39  0.73  14835.0   5292.0    0.39  0.40
           40      0.30  0.67  14835.0   5292.0    0.49  0.49
           80      0.44  1.03  14835.0   5292.0    0.54  0.54
      240  20     -0.05  0.13  14834.0   5292.0    0.28  0.30
           40     -0.57  0.80  14834.0   5292.0    0.42  0.43
           80     -1.16  1.28  14834.0   5292.0    0.52  0.52
k5    60   20      0.37  0.75   8692.0   3276.0    0.39  0.41
           40      0.29  0.75   8692.0   3276.0    0.50  0.51
           80      0.39  0.34   8692.0   3276.0    0.55  0.54
      240  20     -0.35 -0.09   8692.0   3276.0    0.28  0.30
           40     -0.79  0.93   8692.0   3276.0    0.41  0.43
           80     -2.04  0.87   8692.0   3276.0    0.51  0.52
p0.5  60   20      0.39  0.11  34242.0  11809.0    0.36  0.36
           40      0.03 -0.21  34242.0  11809.0    0.48  0.48
           80      0.45  0.17  34242.0  11809.0    0.55  0.54
      240  20      0.16 -0.08  34235.0  11807.0    0.29  0.29
           40     -0.24 -0.03  34235.0  11807.0    0.44  0.43
           80      0.43  1.08  34235.0  11807.0    0.56  0.56
p0.75 60   20      0.65  0.76  19667.0   7012.0    0.33  0.34
           40      0.35  0.41  19667.0   7012.0    0.46  0.46
           80      0.90  0.97  19667.0   7012.0    0.54  0.53
      240  20      0.57  0.27  19667.0   7012.0    0.25  0.26
           40      0.05  0.47  19667.0   7012.0    0.39  0.40
           80      0.38  2.27  19667.0   7012.0    0.52  0.52
p1    60   20      0.75  0.30  12037.0   4392.0    0.32  0.32
           40      0.52  0.45  12037.0   4392.0    0.45  0.45
           80      1.02  1.39  12037.0   4392.0    0.53  0.53
      240  20      0.58 -0.22  12037.0   4392.0    0.23  0.23
           40     -0.14  0.36  12037.0   4392.0    0.37  0.38
           80     -0.21  2.54  12037.0   4392.0    0.50  0.51

In [5]:
if not SYNTHETIC:
    (OUT/'04_decision.json').write_text(json.dumps(res,indent=1,default=float))
    (OUT/'04_manifest.json').write_text(json.dumps({'registration':vx.registration(xp),'runtime_s':time.time()-T0,'trades':int(len(R))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

total 238s
